# Stage 0 — Hypothesis: Asian Session Extreme Fade

**Date locked:** 2026-05-24  
**Author:** Anthony Chung  
**Version:** v1.0  
**Status:** ACTIVE  
**Symbol scope (Phase 1):** BTCUSDT  
**Timeframe:** 1h  
**Session window:** UTC 00:00 – 13:00

---

## Thesis

At the close of the Asian trading session (UTC 13:00), prices that have travelled to the extremes of the Asian range exhibit short-term mean reversion as London / NY liquidity arrives. The hypothesis is that **fading the extreme of the Asian range at session close produces positive expected return after HORROR-mode costs**, driven by overextended positioning from thin Asian liquidity reverting as deeper books arrive.

## Market Inefficiency Targeted

The targeted inefficiency is **end-of-session liquidity-transition overreaction**:

1. The Asian session (UTC 00–13) has comparatively thin BTCUSDT order books vs the London / NY sessions.
2. Thin books amplify the impact of directional flow — prices over-extend relative to their information content.
3. As London (UTC 07+) and NY (UTC 13+) trader populations arrive with deeper books, overextended prices revert.
4. UTC 13:00 is the convergence point — overextended Asian moves meet fresh deep liquidity, and the post-close direction is biased back toward the Asian range midpoint.

## Theoretical and Empirical Support

| Reference | Contribution |
|-----------|--------------|
| De Bondt & Thaler (1985), *Does the Stock Market Overreact?* | Established short-term reversal after extreme moves as a robust cross-asset anomaly |
| Lo & MacKinlay (1990), *When Are Contrarian Profits Due to Stock Market Overreaction?* | Formalized the autocorrelation structure underlying short-horizon mean reversion |
| Madhavan & Smidt (1993), *An Analysis of Changes in Specialist Inventories and Quotations* | Intraday liquidity-induced price reversion; session boundaries as inflection points |
| Heston, Korajczyk & Sadka (2010), *Intraday Patterns in the Cross-section of Stock Returns* | Documented intraday seasonality patterns directly analogous to the proposed session window |

**Crypto-specific reasoning:** high retail participation, 24/7 trading, and concentration of Asian retail volume into specific UTC hours make session-driven liquidity transitions structurally sharper than in traditional equity markets where session boundaries coincide with venue closes.

## Strategy Specification

### Asian Session Definition
- **Window:** UTC 00:00 to UTC 13:00 (13 hours)
- **Asian high:** `max(high)` of all H1 bars in window
- **Asian low:** `min(low)` of all H1 bars in window
- **Asian range:** `Asian_high - Asian_low`
- **Reference price:** close of the UTC 13:00 bar

### Entry Rules (evaluated at UTC 13:00 close)

1. **Range filter:** Asian range must be ≥ `min_range_pct` of session-open price. Filters out low-volatility days where no extreme exists to fade.
2. **Extreme proximity:**
   - Define `distance_to_high = (Asian_high – close_13) / Asian_range`
   - Define `distance_to_low = (close_13 – Asian_low) / Asian_range`
3. **Direction:**
   - If `distance_to_high ≤ extreme_proximity_pct` → enter **SHORT** at UTC 14:00 open
   - If `distance_to_low ≤ extreme_proximity_pct` → enter **LONG** at UTC 14:00 open
   - If both or neither → **no trade** (avoids ambiguous setups)

### Exit Rules (in priority order)
1. **Take profit:** Asian range midpoint (50% retracement of the Asian range). Primary exit.
2. **Stop loss:** range extreme ± `stop_buffer_pct` of price (above Asian high for shorts, below Asian low for longs).
3. **Time stop:** position closed at UTC 21:00 if neither TP nor SL hit (covers full London + early NY window).

### Position Sizing
- Fixed-fractional: 1% of equity at risk per trade, sized via the stop distance.
- Maximum 1 concurrent position. No pyramiding.

### Cost Assumption
- All headline numbers use HORROR mode (see [METHODOLOGY.md](../METHODOLOGY.md)): 0.20% slippage + 0.12% taker fee.
- Round-trip cost ≈ 0.64% of notional.

## Locked Parameter Search Space (Stage 2)

Stage 2 in-sample optimization will search **exactly** the following parameter grid. The grid is locked at the time of this commit and cannot be extended without invalidating Stage 5 holdout claims.

| Parameter | Description | Search values |
|-----------|-------------|---------------|
| `min_range_pct` | Minimum Asian range as fraction of session-open price | [0.005, 0.010, 0.015, 0.020, 0.025] |
| `extreme_proximity_pct` | Closeness to extreme required for entry | [0.15, 0.20, 0.25, 0.30, 0.35] |
| `stop_buffer_pct` | Stop-loss buffer beyond Asian extreme | [0.0025, 0.005, 0.0075, 0.010, 0.015] |
| `tp_fraction` | TP as fraction of Asian range traveled from entry | [0.4, 0.5, 0.6, 0.7] |
| `time_stop_hours` | Hours from entry until forced flat | [4, 6, 8] |

**Total trials:** 5 × 5 × 5 × 4 × 3 = **1,500**

This `N_trials = 1500` is the value passed to the Deflated Sharpe Ratio computation in Stage 5. Recording it here forecloses the option of retrofitting a smaller N to inflate the DSR.

## Falsifiable Predictions

If the thesis is correct, **all** of the following must hold. Any failure terminates the strategy with a postmortem entry in `results/POSTMORTEM.md`.

| # | Test | Threshold | Stage |
|---|------|-----------|-------|
| 1 | Profit Factor on Holdout | > 1.3 | Stage 5 |
| 2 | Sharpe Ratio on Holdout | > 1.0 | Stage 5 |
| 3 | Max Drawdown on Holdout | < 30% | Stage 5 |
| 4 | Deflated Sharpe Ratio (N=1500) | > 0 | Stage 5 |
| 5 | Walk-forward profitable months | ≥ 80% | Stage 4 |
| 6 | Multi-coin replication | ≥ 4 of 6 coins pass tests 1–4 | Stage 6 |
| 7 | IS → OOS-1 PF decay | < 30% drop | Stage 3 |

### Secondary structural prediction
If the inefficiency is real, the **distribution of exits** should be biased toward TP at the range midpoint, not the stop side. A strategy whose winners cluster near stop-adjacent prices (rare wins, deep losses) indicates the thesis is corrupted even if headline PF is positive.

## Data Scope (Phase 1)

| Aspect | Value |
|--------|-------|
| Symbol | BTCUSDT (Binance USDT-margined perpetual) |
| Timeframe | 1h |
| Date range | 2022-01-01 to 2026-03-31 (~4.25 years, ~37,200 bars) |
| IS partition | 2022-01-01 to 2024-07-31 (60%) |
| OOS-1 partition | 2024-08-01 to 2025-07-31 (20%) |
| Holdout partition | 2025-08-01 to 2026-03-31 (20%) |

**Stage 6 multi-coin extension:** ETHUSDT, SOLUSDT, AVAXUSDT, XRPUSDT, BNBUSDT — same date range, same parameter set as chosen in Stage 3.

## Pre-Commitment Statement

> By committing this notebook, I declare that I have **not** examined any backtest of this specification prior to this commit. The parameter search ranges above are a-priori judgments derived from the theoretical framework, not reverse-engineered from observed performance.
>
> Any deviation from this specification in subsequent stages requires a new versioned hypothesis notebook (`v2.0`, `v3.0`, ...) with an explicit acknowledgment that the deviation invalidates any holdout claim derived from the previous version.
>
> — Anthony Chung, 2026-05-24